In [1]:
from pathlib import Path
import pandas as pd
from pathlib import Path
import datetime
from loguru import logger
import numpy as np
import copy

import combined_cooler
import matlab

from phd_visualizations.textual import generate_latex_table
from phd_visualizations.calculations import calculate_metrics

%reload_ext autoreload
%autoreload 2


data_path: Path = Path("../data/")

model_type = "physical"
var_ids = ["Tdc_out", "Twct_out", "Cw", "Tc_out", "Tc_in"]
var_components = ["dc", "wct", "wct", "sc", "sc"]
var_labels = [
    "T$_{dc,out}$ ($^\\circ$C)",
    "T$_{wct,out}$ ($^\\circ$C)",
    "C$_{w}$ (l/h)",
    "T$_{c,out}$ ($^\\circ$C)",
    "T$_{c,in}$ ($^\\circ$C)",
]

assert len(var_ids) == len(var_components), "meh"

# Collect all metrics of interest
metrics_of_interest = ['r2', 'mae', 'mape']

# Latex table parameters
regular_col_ids = [
    "variable",
]

regular_col_labels = [
    r"Predicted\\ variable",
]

group_row_ids = None
# [
#     "variable",
#     "modelling_alternative",
#     "test_id",
#     "sample_time"
# ]

metric_info = {
    "r2": r"R$^2$\\ (-)",
    "mae": r"MAE\\ (s.u.)",
    "mape": r"MAPE\\ (\%)",
    # "time": r"Time\\ (s)",
}

submetric_ids = ["component", "complete"]
submetric_labels = ["Cnt", "CC"]
group_submetric_ids = None # ["avg_val"]

"""

Latex table generation expected input format:
    
    data = [
        {
            "variable": "T$_{dc,out}$ ($^\\circ$C)", -> component = dc
            "metrics": {
                "r2": {"component": "0.98", "complete": "0.97"},
                "rmse": {"component": "0.50", "complete": "0.52"},
                "mae": {"component": "0.45", "complete": "0.48"},
            }
        },
        {
            "variable": "T$_{wct,out}$ ($^\\circ$C)", -> component = wct
            "metrics": {...}
        },
        {
            "variable": "C$_{w}$ (l/h)", -> component = wct
            "metrics": {...}
        },
        {
            "variable": "T$_{c,out}$ ($^\\circ$C)", -> component = sc
            "metrics": {...}
        },
        {
            "variable": "T$_{c,in}$ ($^\\circ$C)", -> component = sc
            "metrics": {...}
        },
    ]

    benchmark_output = [
        eval_date_str : {
            "system": [
                "test_id": "YYYYMMDD",
                "alternative": "",
                "metrics": {},
                "metrics_per_variable": {
                    "var": {}
                },
                "elapsed_time": ,
            ] 
        }
    ]

"""


'\n\nLatex table generation expected input format:\n\n    data = [\n        {\n            "variable": "T$_{dc,out}$ ($^\\circ$C)", -> component = dc\n            "metrics": {\n                "r2": {"component": "0.98", "complete": "0.97"},\n                "rmse": {"component": "0.50", "complete": "0.52"},\n                "mae": {"component": "0.45", "complete": "0.48"},\n            }\n        },\n        {\n            "variable": "T$_{wct,out}$ ($^\\circ$C)", -> component = wct\n            "metrics": {...}\n        },\n        {\n            "variable": "C$_{w}$ (l/h)", -> component = wct\n            "metrics": {...}\n        },\n        {\n            "variable": "T$_{c,out}$ ($^\\circ$C)", -> component = sc\n            "metrics": {...}\n        },\n        {\n            "variable": "T$_{c,in}$ ($^\\circ$C)", -> component = sc\n            "metrics": {...}\n        },\n    ]\n\n    benchmark_output = [\n        eval_date_str : {\n            "system": [\n                "tes

In [ ]:
df_ref


,Tv,Tamb,HR,qc,Tc_in,Ce_cc,Ce,Tcond,Tdc_in,Twct_in,...,Cw,mv,Tdc_out,Twct_out,wdc,wwct,qdc,qwct,Rp,Rs
date,,,,,,,,,,,,,,,,,,,,,
19-Feb-2025,36.261009,17.671711,43.180898,23.998493,28.121629,0.450381,3.653911,35.805044,33.494083,33.172163,...,95.355613,224.076934,28.047818,27.778468,31.974352,22.470002,12.417720,12.259674,0.483,0.054
10-Feb-2025,37.774713,13.531843,61.437262,24.000376,27.700450,0.432014,3.442630,37.052174,34.112194,34.139930,...,108.219547,276.536959,27.430885,27.571514,30.637523,23.770823,12.005001,11.779023,0.500,0.000
12-Feb-2025,36.241372,19.685473,23.739778,23.999315,28.145370,0.381511,4.625762,35.705996,21.977082,33.433644,...,214.507034,217.364415,20.147901,27.961559,0.000000,39.757368,0.235675,23.694971,0.990,0.000
19-Feb-2025,37.575024,19.473600,38.711528,23.998248,27.738758,0.667380,4.861102,36.284572,31.971167,34.366886,...,239.035325,263.510629,23.391675,27.584572,0.000000,50.849622,0.390683,23.691506,0.984,0.201
08-Jul-2025,37.389592,27.213752,50.149914,23.999880,28.115154,7.107880,10.170240,36.956869,34.384180,34.430724,...,156.759557,258.600484,30.136470,24.924979,100.000000,66.439999,13.998713,9.773220,0.417,0.000
08-Jul-2025,38.576245,28.395690,42.827841,24.001603,28.341443,7.863237,10.932670,38.064432,35.242784,35.276181,...,221.497920,270.261573,31.153257,24.004598,100.000000,81.709604,14.000688,9.770016,0.417,0.000
09-Jul-2025,42.399588,27.275200,62.124991,24.000049,35.196531,0.383408,4.809472,42.201587,25.526867,40.275426,...,198.153043,200.822004,25.948971,35.006526,0.000000,39.846699,0.119232,23.694192,0.995,0.000
09-Jul-2025,39.453322,31.500935,41.542744,23.999335,28.792534,8.163791,11.842760,38.904821,35.742446,35.770194,...,263.309141,280.806774,30.781717,27.996002,100.000000,86.901059,5.695932,18.024022,0.763,0.000
13-Jan-2025,44.360271,12.164853,48.714282,24.002221,37.228958,0.271859,3.893380,43.848984,42.061851,21.925185,...,0.000000,214.037281,36.973783,15.001290,30.000000,0.000000,23.798313,0.000540,0.008,0.000


In [ ]:
df_ref = pd.read_csv(data_path / "cc_out_exp.csv", index_col=0)
# Just evaluate the models here to get their outputs given the inputs from reference data

index = pd.to_datetime(df_ref["date"]).values

df_ref.reindex(np.arange(len(df_ref)))

display(df_ref)
display(index)


ValueError: cannot reindex on an axis with duplicate labels

In [ ]:
cc_model = combined_cooler.initialize()

print(f"Using MATLAB package: {cc_model.name}")

params = copy.deepcopy(cc_model.default_parameters(nargout=1))
if model_type == "physical":
    
    params["dc_lb"] = matlab.double([[5.0600,   10.0, 5.2211, 11]])
    params["dc_ub"] = matlab.double([[50.7500,   50.0, 24.1543, 99.1800]])

    # WCT           "Tamb",     "HR",    "Tin",      "q",     "w_fan"
    params["wct_lb"] = matlab.double([[0.1,    0.1,     5.0,    5.0,       0.]])
    params["wct_ub"] = matlab.double([[50.0,   99.99,   55.0,   24.8400,  95.]])
elif model_type == "data":    
    params["dc_model_data_path"] = "/workspaces/SOLhycool/modeling/matlab/component_models/dc_model_data.mat"
    params["wct_model_data_path"] = "/workspaces/SOLhycool/modeling/matlab/component_models/wct_model_data.mat"

results_cc: list[dict] = []
Tc_in = []
Tc_out = []
Tdc_out = []
Ce_dc = []
Twct_out = []
Ce_wct = []
Cw_wct = []
for idx, ds in df_ref.iterrows():

    # Combined cooler
    qc_m3h = matlab.double([ds["qc"]])
    qdc_m3h = matlab.double([ds["qdc"]])
    qwct_m3h = matlab.double([ds["qwct"]])

    Rp, Rs = cc_model.flows_to_ratios(qc_m3h, qdc_m3h, qwct_m3h, nargout=2)

    Tamb_C = matlab.double([ds["Tamb"]])
    HR_pp = matlab.double([ds["HR"]])
    mv_kgh = matlab.double([ds["mv"]])
    wdc = matlab.double([ds["wdc"]])
    wwct = matlab.double([ds["wwct"]])
    Tv = matlab.double([ds["Tv"]])

    # Create the 'options' struct as a Python dictionary
    options = {
        'model_type': model_type,
        'parameters': params,  # Empty struct in MATLAB
        'lb': [], 'ub': [], 'x0': [], 'silence_warnings': True
    }

    Ce_kWe, Cw_lh, detailed = cc_model.combined_cooler_model(Tamb_C, HR_pp, mv_kgh, qc_m3h, Rp, Rs, wdc, wwct, Tv, options, nargout=3)
    results_cc.append(detailed)
    
    # Surface condenser
    mv_kgs = matlab.double([ds["mv"]/3600])  # Convert from kg/h to kg/s
    mc_kgs = matlab.double([ds["qc"]/3.6])  # Convert from m3/h to kg/s (assuming water density ~1000 kg/m3)
    Tv_C = matlab.double([ds["Tv"]])
    
    tc_in, tc_out = cc_model.condenser_model(mv_kgs, Tv_C, mc_kgs, nargout=2)
    Tc_in.append(tc_in)
    Tc_out.append(tc_out)
    
    # Dry cooler
    qdc_m3h = matlab.double([ds["qdc"]])
    wdc = matlab.double([ds["wdc"]])
    Tamb_C = matlab.double([ds["Tamb"]])
    Tin_C = matlab.double([ds["Tdc_in"]])
    
    tdc_out, ce_dc = cc_model.dc_model_physical(Tamb_C, Tin_C, qdc_m3h, wdc, nargout=2)
    Tdc_out.append(tdc_out)
    Ce_dc.append(ce_dc)
    
    # Wet cooler
    qwct_m3h = matlab.double([ds["qwct"]])
    wwct = matlab.double([ds["wwct"]])
    Tamb_C = matlab.double([ds["Tamb"]])
    Tin_C = matlab.double([ds["Twct_in"]])
    HR_pp = matlab.double([ds["HR"]])
    
    twct_out, ce_wct, cw_wct = cc_model.wct_model_physical(Tamb_C, HR_pp, Tin_C, qwct_m3h, wwct, nargout=3)
    Twct_out.append(twct_out)
    Ce_wct.append(ce_wct)
    Cw_wct.append(cw_wct)

# results_dfs.append( pd.DataFrame(results_cc) )
df_mod_cc = pd.DataFrame(results_cc)
df_mod_sc = pd.DataFrame({
    "Tv": df_ref["Tv"].values,
    "Tc_in": Tc_in,
    "Tc_out": Tc_out,
    "qc": df_ref["qc"].values,
    "mv": df_ref["mv"].values,
})
df_mod_dc = pd.DataFrame({
    "Tamb": df_ref["Tamb"].values,
    "Tdc_out": Tdc_out,
    "Ce_dc": Ce_dc,
    "qdc": df_ref["qdc"].values,
    "wdc": df_ref["wdc"].values,
})
df_mod_wct = pd.DataFrame({
    "Tamb": df_ref["Tamb"].values,
    "HR": df_ref["HR"].values,
    "Twct_out": Twct_out,
    "Ce_wct": Ce_wct,
    "Cw": Cw_wct,
    "qwct": df_ref["qwct"].values,
    "wwct": df_ref["wwct"].values,
})


Using MATLAB package: combined_cooler


TypeError: unhashable type: 'Series'

In [21]:
# Filter results to remove entries with inactive components
    
df_mod_cc.loc[df_mod_cc["Qdc"] < 1, "Tdc_out"] = np.nan
df_mod_cc.loc[df_mod_cc["Qwct"] < 1, "Twct_out"] = np.nan

df_mod_dc.loc[df_mod_cc["Qdc"] < 1, "Tdc_out"] = np.nan
df_mod_wct.loc[df_mod_cc["Qwct"] < 1, "Twct_out"] = np.nan

df_ref.loc[df_mod_cc["Qdc"] < 1, "Tdc_out"] = np.nan
df_ref.loc[df_mod_cc["Qwct"] < 1, "Twct_out"] = np.nan

print("Combined cooler:")
display(df_mod_cc)
print("Surface condenser:")
display(df_mod_sc)
print("Dry cooler:")
display(df_mod_dc)
print("Wet cooler:")
display(df_mod_wct)


Combined cooler:


,Tamb,HR,mv,qc,Rp,Rs,wdc,wwct,Ce,Cw,...,Tdc_in,Tdc_out,Ce_dc,Qdc,qwct,Twct_in,Twct_out,Ce_wct,Cw_wct,Qwct
0,17.671711,43.180898,224.076934,23.998493,0.483,0.054,31.974352,22.470002,5.096935,95.530043,...,33.159526,28.069098,0.312366,73.326200,12.261262,32.881371,27.294913,0.138014,95.530043,79.527473
1,13.531843,61.437262,276.536959,24.000376,0.500,0.000,30.637523,23.770823,5.079493,111.887307,...,33.821712,27.215875,0.284241,92.034140,12.000188,33.821712,26.969546,0.147773,111.887307,95.466819
2,19.685473,23.739778,217.364415,23.999315,0.990,0.000,0.000000,39.757368,5.028469,195.773397,...,33.278941,NaN,0.000000,0.000000,23.759322,33.278941,27.600132,0.381511,195.773397,156.648623
3,19.473600,38.711528,263.510629,23.998248,0.984,0.201,0.000000,50.849622,5.313815,225.389836,...,33.888814,NaN,0.000000,0.000000,23.691455,33.888814,27.224474,0.667380,225.389836,183.307771
4,27.213752,50.149914,258.600484,23.999880,0.417,0.000,100.000000,66.439999,11.755116,149.206450,...,33.771250,29.688823,5.867400,66.312809,10.007950,33.771250,24.469662,1.240480,149.206450,108.088079
5,28.395690,42.827841,270.261573,24.001603,0.417,0.000,100.000000,81.709604,12.511319,182.263510,...,34.989055,30.877860,5.867400,66.780750,10.008669,34.989055,23.918692,1.995837,182.263510,128.647991
6,27.275200,62.124991,200.822004,24.000049,0.995,0.000,0.000000,39.846699,5.030727,219.390853,...,40.385228,NaN,0.000000,0.000000,23.880049,40.385228,34.241805,0.383408,219.390853,170.277363
7,31.500935,41.542744,280.806774,23.999335,0.763,0.000,100.000000,86.901059,12.810758,241.868112,...,35.843477,32.158253,5.867400,24.331311,18.311492,35.843477,27.914413,2.296391,241.868112,168.555730
8,12.164853,48.714282,214.037281,24.002221,0.008,0.000,30.000000,0.000000,4.920243,0.000000,...,42.299130,36.823954,0.271859,151.307410,0.192018,42.299130,NaN,0.000000,0.000000,0.000000
9,20.923333,42.056077,303.464304,17.998552,0.346,0.000,59.506664,24.675237,4.204095,119.041564,...,39.959556,29.958103,1.791859,136.652312,6.227499,39.959556,26.671981,0.155379,119.041564,96.056359


Surface condenser:


,Tv,Tc_in,Tc_out,qc,mv
0,36.261009,27.525112,32.921388,23.998493,224.076934
1,37.774713,26.852619,33.501979,24.000376,276.536959
2,36.241372,27.819230,33.053773,23.999315,217.364415
3,37.575024,27.255900,33.593833,23.998248,263.510629
4,37.389592,27.262620,33.483153,23.999880,258.600484
5,38.576245,28.213192,34.706241,24.001603,270.261573
6,42.399588,35.582865,40.399588,24.000049,200.822004
7,39.453322,28.825727,35.566946,23.999335,280.806774
8,44.360271,37.226856,42.360271,24.002221,214.037281
9,43.324459,30.157146,39.833613,17.998552,303.464304


Dry cooler:


,Tamb,Tdc_out,Ce_dc,qdc,wdc
0,17.671711,28.292938,0.312366,12.417720,31.974352
1,13.531843,27.413939,0.284241,12.005001,30.637523
2,19.685473,NaN,0.000000,0.235675,0.000000
3,19.473600,NaN,0.000000,0.390683,0.000000
4,27.213752,29.920168,5.867400,13.998713,100.000000
5,28.395690,30.973380,5.867400,14.000688,100.000000
6,27.275200,NaN,0.000000,0.119232,0.000000
7,31.500935,32.142960,5.867400,5.695932,100.000000
8,12.164853,36.627468,0.271859,23.798313,30.000000
9,20.923333,30.018559,1.791859,11.778898,59.506664


Wet cooler:


,Tamb,HR,Twct_out,Ce_wct,Cw,qwct,wwct
0,17.671711,43.180898,27.478750,0.138014,97.351025,12.259674,22.470002
1,13.531843,61.437262,24.432827,0.147773,147.989133,11.779023,23.770823
2,19.685473,23.739778,27.687677,0.381511,197.492293,23.694971,39.757368
3,19.473600,38.711528,27.498227,0.667380,232.195464,23.691506,50.849622
4,27.213752,50.149914,24.529100,1.240480,154.194789,9.773220,66.439999
5,28.395690,42.827841,23.863365,1.995837,183.134541,9.770016,81.709604
6,27.275200,62.124991,34.145342,0.383408,217.216941,23.694192,39.846699
7,31.500935,41.542744,27.816433,2.296391,239.197971,18.024022,86.901059
8,12.164853,48.714282,NaN,0.000000,0.000000,0.000540,0.000000
9,20.923333,42.056077,26.482780,0.155379,118.305504,6.036892,24.675237


In [46]:
# Build table data
metrics_dict = {
    "complete": {},
    "component": {}
}
table_data = []
for var_id, var_component, var_label in zip(var_ids, var_components, var_labels):
        
    metrics_dict_complete = calculate_metrics(
        metrics=metrics_of_interest,
        y_true=df_ref[var_id].values,
        y_pred=df_mod_cc[var_id].values
    )
    
    if var_component == "dc":
        df_mod = df_mod_dc
    elif var_component == "wct":
        df_mod = df_mod_wct
    elif var_component == "sc":
        df_mod = df_mod_sc
    else:
        raise ValueError(f"Unknown component: {var_component}")
    
    metrics_dict_component = calculate_metrics(
        metrics=metrics_of_interest,
        y_true=df_ref[var_id].values,
        y_pred=df_mod[var_id].values
    )
    
    table_data.append(
        {
            "variable": var_label,
            "metrics": {
                metric_id: {
                    "complete":metrics_dict_complete[metric_id],
                    "component":metrics_dict_component[metric_id]
                } 
                for metric_id in metrics_of_interest
            }
        }
    )
    
table_data


[{'variable': 'T$_{dc,out}$ ($^\\circ$C)',
  'metrics': {'r2': {'complete': np.float64(0.9836528864167545),
    'component': np.float64(0.9883620714736132)},
   'mae': {'complete': np.float64(0.3823347511373418),
    'component': np.float64(0.2898784283365269)},
   'mape': {'complete': np.float64(1.1661879707855058),
    'component': np.float64(0.9016124135595196)}}},
 {'variable': 'T$_{wct,out}$ ($^\\circ$C)',
  'metrics': {'r2': {'complete': np.float64(0.9390868087666555),
    'component': np.float64(0.9245048982439383)},
   'mae': {'complete': np.float64(0.9032440348979622),
    'component': np.float64(1.0058340570923854)},
   'mape': {'complete': np.float64(2.7232231925754196),
    'component': np.float64(3.013258220111358)}}},
 {'variable': 'C$_{w}$ (l/h)',
  'metrics': {'r2': {'complete': np.float64(0.8218995589163411),
    'component': np.float64(0.8688102347787524)},
   'mae': {'complete': np.float64(19.137189235303087),
    'component': np.float64(16.55394214896242)},
   'mape

In [47]:
# Generate table
table_str = generate_latex_table(
    regular_col_ids,
    regular_col_labels,
    metric_info,
    table_data,
    submetric_ids,
    group_row_ids,
    submetric_labels,
    group_submetric_ids
)

print(table_str)


\begin{tabular}{ccccccccccccc}
\hline
\multirow{3}{*}{\textbf{\begin{tabular}[c]{@{}c@{}}Predicted\\ variable\end{tabular}}} &  & \multicolumn{11}{c}{\textbf{Performance metric}}
\\\cline{3-13}
 &  & \multicolumn{3}{c}{\textbf{\begin{tabular}[c]{@{}c@{}}R$^2$\\ (-)\end{tabular}}} &  & \multicolumn{3}{c}{\textbf{\begin{tabular}[c]{@{}c@{}}MAE\\ (s.u.)\end{tabular}}} &  & \multicolumn{3}{c}{\textbf{\begin{tabular}[c]{@{}c@{}}MAPE\\ (\%)\end{tabular}}}
 \\\cline{3-5}\cline{7-9}\cline{11-13}
 &  & Cnt &  & CC &  & Cnt &  & CC &  & Cnt &  & CC
 \\\cline{1-1}\cline{3-3}\cline{5-5}\cline{7-7}\cline{9-9}\cline{11-11}\cline{13-13}
T$_{dc,out}$ ($^\circ$C) &  & 0.99 &  & 0.98 &  & 0.29 &  & 0.38 &  & 0.90 &  & 1.17 \\
T$_{wct,out}$ ($^\circ$C) &  & 0.92 &  & 0.94 &  & 1.01 &  & 0.90 &  & 3.01 &  & 2.72 \\
C$_{w}$ (l/h) &  & 0.87 &  & 0.82 &  & 16.55 &  & 19.14 &  & 10.42 &  & 11.03 \\
T$_{c,out}$ ($^\circ$C) &  & 0.98 &  & 0.99 &  & 0.60 &  & 0.41 &  & 1.51 &  & 1.00 \\
T$_{c,in}$ ($^\circ$C) & 